In [2]:
#!/usr/bin/env python3

from pathlib import Path
import math
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go


# =============================================================================
# Paths
# =============================================================================

BASE = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/TelomereHunter2"
)

RESULTS_ROOT = BASE / "results"
OUTPUT_DIR = BASE / "cohort_results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Helper functions
# =============================================================================

def get_numeric_value(row, possible_columns, required=True):
    """
    Retrieve a numeric value from the first available column name.

    This supports differences between TelomereHunter2 summary versions,
    such as read_lengths versus read_length.
    """

    for column in possible_columns:

        if column not in row.index:
            continue

        value = row[column]

        if pd.isna(value):
            continue

        value = str(value).strip()

        if value == "":
            continue

        return float(value)

    if required:
        raise ValueError(
            "Missing required column. Checked: "
            + ", ".join(possible_columns)
        )

    return np.nan


def save_figure(figure, output_name):
    """Save a Plotly figure as HTML, PNG and PDF."""

    html_file = OUTPUT_DIR / "{}.html".format(output_name)
    png_file = OUTPUT_DIR / "{}.png".format(output_name)
    pdf_file = OUTPUT_DIR / "{}.pdf".format(output_name)

    figure.write_html(str(html_file))

    try:
        figure.write_image(str(png_file), scale=2)
        figure.write_image(str(pdf_file))

        print("Saved:", png_file)
        print("Saved:", pdf_file)

    except Exception as error:
        print(
            "WARNING: PNG/PDF export failed: {}".format(error),
            file=sys.stderr
        )
        print(
            "The interactive HTML figure was still generated.",
            file=sys.stderr
        )

    print("Saved:", html_file)


# =============================================================================
# Locate and process summary files
# =============================================================================

if not RESULTS_ROOT.is_dir():
    raise FileNotFoundError(
        "Results directory does not exist: {}".format(RESULTS_ROOT)
    )

results = []
summary_files_found = 0

for pair_directory in sorted(RESULTS_ROOT.iterdir()):

    if not pair_directory.is_dir():
        continue

    pair_id = pair_directory.name

    # Ignore archived outputs from failed or incomplete runs.
    if ".failed_" in pair_id or ".incomplete_" in pair_id:
        continue

    summary_file = pair_directory / "{}_summary.tsv".format(pair_id)

    if not summary_file.is_file():
        continue

    if summary_file.stat().st_size == 0:
        print(
            "WARNING: Empty summary skipped: {}".format(summary_file),
            file=sys.stderr
        )
        continue

    summary_files_found += 1

    try:
        # Read every column as text first.
        summary = pd.read_csv(
            str(summary_file),
            sep="\t",
            dtype=str
        )

        # Remove whitespace from column names.
        summary.columns = [
            str(column).strip()
            for column in summary.columns
        ]

        if "sample" not in summary.columns:
            raise ValueError("The 'sample' column is missing")

        summary["sample_clean"] = (
            summary["sample"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )

        # Select only biological sample rows.
        tumor_rows = summary[
            summary["sample_clean"] == "tumor"
        ]

        control_rows = summary[
            summary["sample_clean"] == "control"
        ]

        if len(tumor_rows) != 1:
            raise ValueError(
                "Expected one tumor row; found {}".format(
                    len(tumor_rows)
                )
            )

        if len(control_rows) != 1:
            raise ValueError(
                "Expected one control row; found {}".format(
                    len(control_rows)
                )
            )

        tumor = tumor_rows.iloc[0]
        control = control_rows.iloc[0]

        # ---------------------------------------------------------------------
        # Primary telomere measurements
        # ---------------------------------------------------------------------

        tumor_tel_content = get_numeric_value(
            tumor,
            ["tel_content"]
        )

        normal_tel_content = get_numeric_value(
            control,
            ["tel_content"]
        )

        if tumor_tel_content <= 0 or normal_tel_content <= 0:
            raise ValueError(
                "Tumor and normal tel_content must be greater than zero"
            )

        # ---------------------------------------------------------------------
        # Sequencing and telomeric-read counts
        # ---------------------------------------------------------------------

        tumor_total_reads = int(
            get_numeric_value(
                tumor,
                ["total_reads"]
            )
        )

        normal_total_reads = int(
            get_numeric_value(
                control,
                ["total_reads"]
            )
        )

        tumor_telomeric_reads = int(
            get_numeric_value(
                tumor,
                ["tel_read_count"]
            )
        )

        normal_telomeric_reads = int(
            get_numeric_value(
                control,
                ["tel_read_count"]
            )
        )

        if tumor_total_reads <= 0 or normal_total_reads <= 0:
            raise ValueError(
                "Total-read counts must be greater than zero"
            )

        # Calculate TRPM
        tumor_trpm = (
            tumor_telomeric_reads / tumor_total_reads
        ) * 1_000_000

        normal_trpm = (
            normal_telomeric_reads / normal_total_reads
        ) * 1_000_000

        # ---------------------------------------------------------------------
        # Additional metadata
        # ---------------------------------------------------------------------

        read_length = get_numeric_value(
            tumor,
            ["read_lengths", "read_length"],
            required=False
        )

        repeat_threshold = get_numeric_value(
            tumor,
            ["repeat_threshold_used"],
            required=False
        )

        # ---------------------------------------------------------------------
        # Calculate tumor-to-normal telomere change
        # ---------------------------------------------------------------------

        tumor_normal_ratio = (
            tumor_tel_content /
            normal_tel_content
        )

        log2_ratio = math.log2(
            tumor_normal_ratio
        )

        percentage_change = (
            tumor_normal_ratio - 1.0
        ) * 100.0

        if np.isclose(log2_ratio, 0.0, atol=1e-12):
            direction = "No change"
        elif log2_ratio < 0:
            direction = "Shortening"
        else:
            direction = "Lengthening"

        # ---------------------------------------------------------------------
        # Extract TelomereHunter2-reported log2 ratio for validation
        #
        # In the log2(tumor/control) row, the result is stored under
        # the tel_content column.
        # ---------------------------------------------------------------------

        log2_rows = summary[
            summary["sample_clean"] == "log2(tumor/control)"
        ]

        th2_reported_log2 = np.nan
        log2_check_difference = np.nan

        if len(log2_rows) >= 1:

            log2_row = log2_rows.iloc[0]

            th2_reported_log2 = get_numeric_value(
                log2_row,
                ["tel_content"],
                required=False
            )

            if not pd.isna(th2_reported_log2):
                log2_check_difference = (
                    log2_ratio -
                    th2_reported_log2
                )

        # ---------------------------------------------------------------------
        # Store pair-level result
        # ---------------------------------------------------------------------

        results.append(
            {
                "Pair_ID": pair_id,
                "Normal_total_reads": normal_total_reads,
                "Tumor_total_reads": tumor_total_reads,
                "Normal_telomeric_reads": normal_telomeric_reads,
                "Tumor_telomeric_reads": tumor_telomeric_reads,
                "Normal_TRPM_calculated": normal_trpm,
                "Tumor_TRPM_calculated": tumor_trpm,
                "Normal_tel_content": normal_tel_content,
                "Tumor_tel_content": tumor_tel_content,
                "Tumor_Normal_ratio": tumor_normal_ratio,
                "log2_Tumor_Normal": log2_ratio,
                "TH2_reported_log2": th2_reported_log2,
                "Log2_check_difference": log2_check_difference,
                "Percentage_change": percentage_change,
                "Direction": direction,
                "Repeat_threshold": repeat_threshold,
                "Read_length": read_length,
                "Summary_file": str(summary_file),
            }
        )

        print("PASS:", pair_id)

    except Exception as error:
        print(
            "FAIL: {}: {}".format(pair_id, error),
            file=sys.stderr
        )


# =============================================================================
# Validate collected results
# =============================================================================

if summary_files_found == 0:
    raise RuntimeError(
        "No TelomereHunter2 summary files were found"
    )

if not results:
    raise RuntimeError(
        "Summary files were found, but none could be parsed"
    )

df = pd.DataFrame(results)

# Shortest relative telomeres first, longest last.
df = df.sort_values(
    "log2_Tumor_Normal",
    ascending=True
).reset_index(drop=True)

# Shorter labels for the waterfall x-axis.
df["Plot_Label"] = df["Pair_ID"].str.replace(
    r"^JC287-",
    "",
    regex=True
)


# =============================================================================
# Save combined results
# =============================================================================

combined_tsv = OUTPUT_DIR / "TelomereHunter2_combined_results.tsv"
combined_csv = OUTPUT_DIR / "TelomereHunter2_combined_results.csv"

df.to_csv(
    str(combined_tsv),
    sep="\t",
    index=False,
    float_format="%.6f"
)

df.to_csv(
    str(combined_csv),
    index=False,
    float_format="%.6f"
)

print("\nCombined results:")
print(combined_tsv)


# =============================================================================
# Cohort statistics
# =============================================================================

n_pairs = len(df)

n_shortening = int(
    (df["log2_Tumor_Normal"] < 0).sum()
)

n_lengthening = int(
    (df["log2_Tumor_Normal"] > 0).sum()
)

n_unchanged = int(
    np.isclose(
        df["log2_Tumor_Normal"],
        0.0,
        atol=1e-12
    ).sum()
)

median_normal = df["Normal_tel_content"].median()
median_tumor = df["Tumor_tel_content"].median()
median_ratio = df["Tumor_Normal_ratio"].median()
median_log2 = df["log2_Tumor_Normal"].median()
median_percent = df["Percentage_change"].median()

q1_log2 = df["log2_Tumor_Normal"].quantile(0.25)
q3_log2 = df["log2_Tumor_Normal"].quantile(0.75)


# =============================================================================
# Wilcoxon signed-rank test
# Tests whether log2 tumor/normal ratios differ from zero.
# =============================================================================

wilcoxon_statistic = np.nan
p_value = np.nan

try:
    from scipy.stats import wilcoxon

    values = df["log2_Tumor_Normal"].to_numpy(dtype=float)

    if not np.allclose(values, 0.0):

        wilcoxon_statistic, p_value = wilcoxon(
            values,
            alternative="two-sided"
        )

except ImportError:
    print(
        "NOTE: SciPy is unavailable; "
        "the Wilcoxon test was skipped."
    )


# =============================================================================
# Save cohort statistics
# =============================================================================

statistics_file = (
    OUTPUT_DIR /
    "TelomereHunter2_cohort_statistics.txt"
)

with statistics_file.open("w") as handle:

    handle.write(
        "Number of summary files found: {}\n".format(
            summary_files_found
        )
    )

    handle.write(
        "Number of evaluable pairs: {}\n".format(
            n_pairs
        )
    )

    handle.write(
        "Tumors with relative shortening: {}\n".format(
            n_shortening
        )
    )

    handle.write(
        "Tumors with relative lengthening: {}\n".format(
            n_lengthening
        )
    )

    handle.write(
        "Tumors with no change: {}\n".format(
            n_unchanged
        )
    )

    handle.write(
        "Median normal tel_content: {:.6f}\n".format(
            median_normal
        )
    )

    handle.write(
        "Median tumor tel_content: {:.6f}\n".format(
            median_tumor
        )
    )

    handle.write(
        "Median tumor/normal ratio: {:.6f}\n".format(
            median_ratio
        )
    )

    handle.write(
        "Median log2 tumor/normal: {:.6f}\n".format(
            median_log2
        )
    )

    handle.write(
        "IQR log2 tumor/normal: {:.6f} to {:.6f}\n".format(
            q1_log2,
            q3_log2
        )
    )

    handle.write(
        "Median percentage change: {:+.2f}%\n".format(
            median_percent
        )
    )

    if not pd.isna(p_value):

        handle.write(
            "Wilcoxon statistic: {:.6f}\n".format(
                wilcoxon_statistic
            )
        )

        handle.write(
            "Wilcoxon signed-rank P value: {:.6g}\n".format(
                p_value
            )
        )

print("Cohort statistics:")
print(statistics_file)


# =============================================================================
# Plot colors
# =============================================================================

NORMAL_COLOR = "#2878B5"
TUMOR_COLOR = "#D64B3C"

SHORTENING_COLOR = "#2878B5"
LENGTHENING_COLOR = "#D64B3C"
NO_CHANGE_COLOR = "#808080"

PAIR_LINE_COLOR = "#A6A6A6"


# =============================================================================
# Figure 1: Paired dot plot
# =============================================================================

paired_figure = go.Figure()

# Connecting lines
for _, row in df.iterrows():

    paired_figure.add_trace(
        go.Scatter(
            x=["Normal", "Tumor"],
            y=[
                row["Normal_tel_content"],
                row["Tumor_tel_content"]
            ],
            mode="lines",
            line=dict(
                color=PAIR_LINE_COLOR,
                width=1.5
            ),
            hoverinfo="skip",
            showlegend=False
        )
    )

# Normal points
paired_figure.add_trace(
    go.Scatter(
        x=["Normal"] * n_pairs,
        y=df["Normal_tel_content"].tolist(),
        mode="markers",
        name="Matched normal",
        customdata=df["Pair_ID"].tolist(),
        marker=dict(
            color=NORMAL_COLOR,
            size=13,
            line=dict(
                color="white",
                width=1
            )
        ),
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "Sample: Matched normal<br>"
            "tel_content: %{y:.3f}"
            "<extra></extra>"
        )
    )
)

# Tumor points
paired_figure.add_trace(
    go.Scatter(
        x=["Tumor"] * n_pairs,
        y=df["Tumor_tel_content"].tolist(),
        mode="markers",
        name="Tumor",
        customdata=df["Pair_ID"].tolist(),
        marker=dict(
            color=TUMOR_COLOR,
            size=13,
            line=dict(
                color="white",
                width=1
            )
        ),
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "Sample: Tumor<br>"
            "tel_content: %{y:.3f}"
            "<extra></extra>"
        )
    )
)

annotation = (
    "Pairs = {}<br>"
    "Shortening = {}<br>"
    "Lengthening = {}<br>"
    "Median change = {:+.1f}%"
).format(
    n_pairs,
    n_shortening,
    n_lengthening,
    median_percent
)

if not pd.isna(p_value):
    annotation += "<br>Wilcoxon P = {:.3g}".format(p_value)

paired_figure.add_annotation(
    x=1.04,
    y=0.98,
    xref="paper",
    yref="paper",
    text=annotation,
    showarrow=False,
    align="left",
    xanchor="left",
    yanchor="top",
    font=dict(
        family="Arial",
        size=15,
        color="black"
    ),
    bordercolor="#A6A6A6",
    borderwidth=1,
    borderpad=8,
    bgcolor="white"
)

paired_figure.update_layout(
    title=dict(
        text=(
            "Relative telomere content in paired "
            "tumor–normal WGS"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=24)
    ),
    template="simple_white",
    width=1050,
    height=800,
    font=dict(
        family="Arial",
        size=18,
        color="black"
    ),
    margin=dict(
        l=110,
        r=270,
        t=130,
        b=100
    ),
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.10,
        yanchor="bottom"
    )
)

paired_figure.update_xaxes(
    title=None,
    tickfont=dict(size=19),
    showline=True,
    linecolor="black",
    linewidth=1.5
)

paired_figure.update_yaxes(
    title="GC-corrected telomere content",
    rangemode="tozero",
    showline=True,
    linecolor="black",
    linewidth=1.5,
    gridcolor="#E5E5E5",
    zeroline=False
)


# =============================================================================
# Figure 2: Waterfall plot
# =============================================================================

bar_colors = []

for value in df["log2_Tumor_Normal"]:

    if np.isclose(value, 0.0, atol=1e-12):
        bar_colors.append(NO_CHANGE_COLOR)
    elif value < 0:
        bar_colors.append(SHORTENING_COLOR)
    else:
        bar_colors.append(LENGTHENING_COLOR)


customdata = list(
    zip(
        df["Pair_ID"].tolist(),
        df["Normal_tel_content"].tolist(),
        df["Tumor_tel_content"].tolist(),
        df["Tumor_Normal_ratio"].tolist(),
        df["Percentage_change"].tolist(),
        df["Direction"].tolist()
    )
)

waterfall_figure = go.Figure()

waterfall_figure.add_trace(
    go.Bar(
        x=df["Plot_Label"].tolist(),
        y=df["log2_Tumor_Normal"].tolist(),
        marker=dict(
            color=bar_colors,
            line=dict(
                color="black",
                width=0.7
            )
        ),
        text=[
            "{:+.1f}%".format(value)
            for value in df["Percentage_change"]
        ],
        textposition="outside",
        cliponaxis=False,
        customdata=customdata,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Normal tel_content: %{customdata[1]:.3f}<br>"
            "Tumor tel_content: %{customdata[2]:.3f}<br>"
            "Tumor/Normal ratio: %{customdata[3]:.3f}<br>"
            "log₂(T/N): %{y:.3f}<br>"
            "Change: %{customdata[4]:+.1f}%<br>"
            "Direction: %{customdata[5]}"
            "<extra></extra>"
        ),
        showlegend=False
    )
)

waterfall_figure.add_hline(
    y=0,
    line_color="black",
    line_width=1.5
)

# Manual legend
waterfall_figure.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            symbol="square",
            size=15,
            color=SHORTENING_COLOR
        ),
        name="Relative shortening"
    )
)

waterfall_figure.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            symbol="square",
            size=15,
            color=LENGTHENING_COLOR
        ),
        name="Relative lengthening"
    )
)

waterfall_figure.update_layout(
    title=dict(
        text=(
            "Tumor-to-normal change in relative "
            "telomere content"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=24)
    ),
    template="simple_white",
    width=1500,
    height=850,
    font=dict(
        family="Arial",
        size=18,
        color="black"
    ),
    margin=dict(
        l=120,
        r=70,
        t=130,
        b=300
    ),
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=1.08,
        yanchor="bottom"
    ),
    bargap=0.25
)

waterfall_figure.update_xaxes(
    title="Matched tumor–normal pair",
    tickangle=-50,
    tickfont=dict(size=14),
    showline=True,
    linecolor="black",
    linewidth=1.5
)

waterfall_figure.update_yaxes(
    title="log₂(Tumor/Normal telomere content)",
    showline=True,
    linecolor="black",
    linewidth=1.5,
    gridcolor="#E5E5E5",
    zeroline=False
)


# =============================================================================
# Save figures
# =============================================================================

save_figure(
    paired_figure,
    "paired_telomere_content"
)

save_figure(
    waterfall_figure,
    "waterfall_log2_tumor_normal"
)


# =============================================================================
# Display results
# =============================================================================

display_columns = [
    "Pair_ID",
    "Normal_tel_content",
    "Tumor_tel_content",
    "Tumor_Normal_ratio",
    "log2_Tumor_Normal",
    "TH2_reported_log2",
    "Percentage_change",
    "Direction"
]

print("\nPer-pair results:\n")

print(
    df[display_columns].to_string(
        index=False,
        float_format=lambda value: "{:.4f}".format(value)
    )
)

print("\n============================================================")
print("Analysis completed")
print("============================================================")
print("Summary files found:      {}".format(summary_files_found))
print("Evaluable pairs:          {}".format(n_pairs))
print("Relative shortening:      {}".format(n_shortening))
print("Relative lengthening:     {}".format(n_lengthening))
print("Median T/N ratio:         {:.3f}".format(median_ratio))
print("Median log2(T/N):         {:.3f}".format(median_log2))
print("Median percentage change: {:+.2f}%".format(median_percent))

if not pd.isna(p_value):
    print("Wilcoxon P value:         {:.6g}".format(p_value))

print("============================================================")


PASS: JC287-TP08-1447_TP08-1335
PASS: JC287-TP08-525_TP08-446
PASS: JC287-TP09-71_TP09-11
PASS: JC287-TP10-705_TP16-M397
PASS: JC287-TP14-385_TP14-385
PASS: JC287-TP14-490_TP14-49
PASS: JC287-TP15-M509_TP15-M509
PASS: JC287-TP16-M241_TP16-M241
PASS: JC287-TP16-M953_TP16-M953
PASS: JC287-TP17-M11_TP17-M11

Combined results:
/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/TelomereHunter2/cohort_results/TelomereHunter2_combined_results.tsv
Cohort statistics:
/ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/TelomereHunter2/cohort_results/TelomereHunter2_cohort_statistics.txt
Saved: /ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/TelomereHunter2/cohort_results/paired_telomere_content.png
Saved: /ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/TelomereHunter2/cohort_results/paired_telomere_content.pdf
Saved: /ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/TelomereHunter2/cohort_results/paired_telomere_content.html
Saved: /ix1/alee/LO_LAB/Personal/Rahul/pILC/WGS/TelomereHunter2/cohort_results/waterfall_log2_tumor_normal.png
Save